# Step 7: Hybrid Recommendation & Evaluation

## Phase 5: Explainable Recommendation

**Components:**
1. **Readiness Scoring**: β₁·concept_coverage + β₂·prerequisite_strength
2. **Hybrid Fusion**: γ₁·ALS + γ₂·rule_confidence + γ₃·R_score + γ₄·readiness
3. **Explanation Generation**: Path-based, Concept-based, Similarity-based
4. **Evaluation**: Precision@K, NDCG@K, MRR, Explanation Coverage

**Formula:**
```
final_score(c) = 0.35·ALS_score + 0.25·rule_confidence + 0.20·R_score + 0.20·readiness
```

With soft constraint: if readiness < 0.5, score *= 0.7

In [1]:

import json
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
import networkx as nx
import csv
from typing import Dict, List, Tuple, Any, Set

OUTPUT_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output"

# Hybrid weights
W_ALS = 0.35
W_RULE = 0.25
W_RSCORE = 0.20
W_READINESS = 0.20

# Readiness threshold
READINESS_THRESHOLD = 0.5
READINESS_PENALTY = 0.7

print("Step 7: Hybrid Recommendation & Evaluation")
print(f"Hybrid weights: ALS={W_ALS}, Rule={W_RULE}, R_score={W_RSCORE}, Readiness={W_READINESS}")

Step 7: Hybrid Recommendation & Evaluation
Hybrid weights: ALS=0.35, Rule=0.25, R_score=0.2, Readiness=0.2


## Load All Data

In [2]:
def load_data() -> Tuple[Dict, Dict, Dict, List, Dict, Dict, Dict]:
    """
    Load all required data files

    Returns:
        Tuple containing all loaded data structures
    """
    print("Loading data...")

    # Load user sequences
    with open(f"{OUTPUT_PATH}/user_sequences.json", "r") as f:
        user_sequences = json.load(f)

    # Convert to dict format: {user_id: [sequence_items]}
    user_sequences_dict = {}
    for record in user_sequences:
        user_id = record["user_id"]
        user_sequences_dict[user_id] = record["sequence"]

    print(f"  - User sequences: {len(user_sequences_dict)}")

    # Load course concepts
    with open(f"{OUTPUT_PATH}/course_concepts.json", "r") as f:
        course_concepts = json.load(f)
    print(f"  - Course concepts: {len(course_concepts)}")

    # Load course prerequisites (instead of concept prerequisites)
    try:
        with open(f"{OUTPUT_PATH}/course_prerequisites.pkl", "rb") as f:
            course_prereqs = pickle.load(f)
        print(f"  - Course prerequisites: {len(course_prereqs)}")
    except FileNotFoundError:
        print("  - Warning: course_prerequisites.pkl not found, using empty dict")
        course_prereqs = {}

    # Load association rules
    try:
        with open(f"{OUTPUT_PATH}/association_rules.pkl", "rb") as f:
            association_rules = pickle.load(f)
        print(f"  - Association rules: {len(association_rules)}")
    except FileNotFoundError:
        with open(f"{OUTPUT_PATH}/association_rules.json", "r") as f:
            association_rules = json.load(f)
        print(f"  - Association rules: {len(association_rules)}")

    # Load ALS embeddings
    with open(f"{OUTPUT_PATH}/als_embeddings.pkl", "rb") as f:
        als_data = pickle.load(f)
        user_embeddings = als_data["user_embeddings"]
        course_embeddings = als_data["course_embeddings"]
    print(f"  - User embeddings: {len(user_embeddings)}")
    print(f"  - Course embeddings: {len(course_embeddings)}")

    # Load R_matrix from CSV
    R_data = {}
    try:
        import csv
        with open(f"{OUTPUT_PATH}/R_matrix/part-00000-359a1dec-eafc-428a-bfce-fbbc2249c1e7-c000.csv", "r", encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                user_id = row["user_id"]
                course_id = row["course_id"]
                r_score = float(row["R_score"])
                R_data[(user_id, course_id)] = r_score
        print(f"  - R_matrix entries: {len(R_data)}")
    except FileNotFoundError:
        print("  - Warning: R_matrix not found, will use default R_score")

    return user_sequences_dict, course_concepts, course_prereqs, association_rules, \
           user_embeddings, course_embeddings, R_data

## Step 7.1: Calculate Readiness Score

In [3]:
def calculate_readiness(user_id: str, target_course: str,
                        user_sequences: Dict, course_concepts: Dict,
                        course_prereqs: Dict) -> Tuple[float, List]:
    """
    Calculate readiness score for user to take target course

    Args:
        user_id: User identifier
        target_course: Course ID to check readiness for
        user_sequences: Dict of user sequences
        course_concepts: Dict of course concepts
        course_prereqs: Dict of course prerequisites {course_a: {course_b: {strength, ...}}}

    Returns:
        Tuple of (readiness_score, covered_concepts_list)
    """
    if user_id not in user_sequences:
        return 0.0, []

    # Get user's completed courses (R_score >= 0.6 means passed)
    user_courses = set()
    mastered_courses_with_score = []
    for item in user_sequences[user_id]:
        # Item format: [sequence_order, course_id, R_score, passed]
        if len(item) >= 4:
            course_id = item[1]
            r_score = item[2]
            passed = item[3]
            user_courses.add(course_id)
            if passed == 1 or r_score >= 0.6:
                mastered_courses_with_score.append(course_id)

    # Get target course concepts
    target_concepts = set(course_concepts.get(target_course, []))

    # Method 1: Check if user has taken any prerequisite courses
    prereq_strength = 0.0
    prereq_courses_matched = []

    for mastered_course in mastered_courses_with_score:
        if mastered_course in course_prereqs:
            if target_course in course_prereqs[mastered_course]:
                strength = course_prereqs[mastered_course][target_course].get("strength", 0)
                prereq_strength = max(prereq_strength, strength)
                prereq_courses_matched.append(mastered_course)

    # Method 2: Calculate concept coverage from user's mastered courses
    mastered_concepts = set()
    for course_id in mastered_courses_with_score:
        mastered_concepts.update(course_concepts.get(course_id, []))

    # Concept coverage
    covered = mastered_concepts & target_concepts
    coverage = len(covered) / len(target_concepts) if target_concepts else 1.0

    # Combined readiness:
    # - 50% from course prerequisite strength (if user took prerequisite course)
    # - 50% from concept coverage (user has relevant concepts)
    readiness = 0.5 * prereq_strength + 0.5 * coverage

    return readiness, sorted(covered)[:10]  # Limit to 10 for display


## Step 7.2: Calculate ALS Score

In [4]:
def calculate_als_score(user_id: str, course_id: str,
                        user_embeddings: Dict, course_embeddings: Dict) -> float:
    """
    Calculate ALS score as dot product of embeddings

    Args:
        user_id: User identifier
        course_id: Course identifier
        user_embeddings: Dict of user embeddings
        course_embeddings: Dict of course embeddings

    Returns:
        Dot product score
    """
    if user_id not in user_embeddings or course_id not in course_embeddings:
        return 0.0

    user_emb = np.array(user_embeddings[user_id])
    course_emb = np.array(course_embeddings[course_id])

    # Dot product
    score = np.dot(user_emb, course_emb)
    return float(score)

## Step 7.3: Find Matching Rules

In [5]:
def find_matching_rules(user_courses: List[str], target_course: str,
                        association_rules: List[Dict]) -> List[Dict]:
    """
    Find association rules where antecedent matches user history

    Args:
        user_courses: List of courses user has taken
        target_course: Target course to recommend
        association_rules: List of association rules

    Returns:
        List of matching rules sorted by confidence
    """
    user_course_set = set(user_courses)
    matching = []

    for rule in association_rules:
        antecedent = set(rule["antecedent"])
        consequent = rule["consequent"]

        # Check if antecedent is subset of user courses and consequent includes target
        if antecedent.issubset(user_course_set) and target_course in consequent:
            matching.append(rule)

    # Sort by confidence
    matching.sort(key=lambda x: x["confidence"], reverse=True)
    return matching


## Step 7.4: Hybrid Scoring

In [6]:
def calculate_hybrid_score(user_id: str, course_id: str, user_courses: List[str],
                           user_embeddings: Dict, course_embeddings: Dict,
                           association_rules: List[Dict], R_data: Dict,
                           user_sequences: Dict, course_concepts: Dict,
                           course_prereqs: Dict) -> Dict[str, Any]:
    """
    Calculate hybrid score for user-course pair

    Args:
        user_id: User identifier
        course_id: Course identifier
        user_courses: List of courses user has taken
        user_embeddings: User embeddings dict
        course_embeddings: Course embeddings dict
        association_rules: Association rules list
        R_data: R_matrix data dict
        user_sequences: User sequences dict
        course_concepts: Course concepts dict
        course_prereqs: Course prerequisites dict

    Returns:
        Dict with all score components
    """
    # ALS score
    als_score = calculate_als_score(user_id, course_id, user_embeddings, course_embeddings)
    als_score_norm = (als_score + 1) / 2  # Normalize to [0, 1]

    # Rule confidence
    matching_rules = find_matching_rules(user_courses, course_id, association_rules)
    rule_confidence = matching_rules[0]["confidence"] if matching_rules else 0.0

    # R_score
    r_score = R_data.get((user_id, course_id), 0.4)
    r_score_norm = r_score / 1.0  # Already in [0, 1]

    # Readiness
    readiness, covered_concepts = calculate_readiness(
        user_id, course_id, user_sequences, course_concepts, course_prereqs
    )

    # Hybrid combination
    final_score = (
        W_ALS * als_score_norm +
        W_RULE * rule_confidence +
        W_RSCORE * r_score_norm +
        W_READINESS * readiness
    )

    # Soft constraint: penalty if readiness < threshold
    if readiness < READINESS_THRESHOLD:
        final_score *= READINESS_PENALTY

    return {
        "user_id": user_id,
        "course_id": course_id,
        "final_score": final_score,
        "als_score": als_score_norm,
        "rule_confidence": rule_confidence,
        "r_score": r_score_norm,
        "readiness": readiness,
        "covered_concepts": covered_concepts
    }


## Step 7.5: Generate Recommendations for All Users

In [7]:
def generate_explanation(user_id: str, course_id: str, user_courses: List[str],
                         score_data: Dict, course_concepts: Dict,
                         association_rules: List[Dict]) -> List[Dict]:
    """
    Generate multi-type explanation for recommendation

    Args:
        user_id: User identifier
        course_id: Course identifier
        user_courses: List of user's courses
        score_data: Score data from hybrid calculation
        course_concepts: Course concepts dict
        association_rules: Association rules list

    Returns:
        List of explanation dicts
    """
    explanations = []

    # Type 1: Path-based
    matching_rules = find_matching_rules(user_courses, course_id, association_rules)
    if matching_rules:
        rule = matching_rules[0]
        antecedent = ", ".join(rule["antecedent"])
        conf = rule["confidence"]
        explanations.append({
            "type": "path-based",
            "text": f"Students who took {antecedent} succeed in this course with {conf:.0%} probability"
        })

    # Type 2: Concept-based
    covered = score_data.get("covered_concepts", [])
    if covered:
        target_concepts = course_concepts.get(course_id, [])
        coverage = len(covered) / len(target_concepts) * 100 if target_concepts else 0
        explanations.append({
            "type": "concept-based",
            "text": f"You've mastered {coverage:.0f}% of prerequisites: {', '.join(covered[:5])}"
        })

    # Type 3: Similarity-based (if ALS is high)
    als_score = score_data.get("als_score", 0)
    if als_score > 0.7:
        explanations.append({
            "type": "similarity-based",
            "text": "Students with similar learning patterns also took this course"
        })

    return explanations


## Step 7.6: Generate Explanations

In [8]:

def generate_recommendations(user_sequences: Dict, course_concepts: Dict,
                             user_embeddings: Dict, course_embeddings: Dict,
                             association_rules: List[Dict], R_data: Dict,
                             course_prereqs: Dict,
                             sample_size: int = 100,
                             top_k: int = 10) -> Tuple[Dict, Dict]:
    """
    Generate recommendations for sample users

    Args:
        user_sequences: User sequences dict
        course_concepts: Course concepts dict
        user_embeddings: User embeddings dict
        course_embeddings: Course embeddings dict
        association_rules: Association rules list
        R_data: R_matrix data dict
        course_prereqs: Course prerequisites dict
        sample_size: Number of users to sample
        top_k: Number of recommendations per user

    Returns:
        Tuple of (recommendations_dict, explanations_dict)
    """
    # Get all courses
    all_courses = list(course_concepts.keys())
    print(f"Total courses: {len(all_courses)}")

    # Sample users
    sample_users = list(user_sequences.keys())[:sample_size]
    print(f"Generating recommendations for {len(sample_users)} users...")

    recommendations = {}
    explanations = {}

    for idx, user_id in enumerate(sample_users):
        if (idx + 1) % 20 == 0:
            print(f"  Processing user {idx + 1}/{len(sample_users)}...")

        # Get user's courses
        user_courses = [item[1] for item in user_sequences[user_id]]

        # Candidate courses (not taken)
        candidate_courses = [c for c in all_courses if c not in user_courses]

        # Limit candidates for speed
        candidate_courses = candidate_courses[:100]

        # Score all candidates
        scores = []
        for course_id in candidate_courses:
            score = calculate_hybrid_score(
                user_id, course_id, user_courses,
                user_embeddings, course_embeddings,
                association_rules, R_data,
                user_sequences, course_concepts,course_prereqs
            )
            scores.append(score)

        # Sort by final score
        scores.sort(key=lambda x: x["final_score"], reverse=True)

        # Top-k recommendations
        recommendations[user_id] = scores[:top_k]

        # Generate explanations for top-5
        user_explanations = []
        for rec in scores[:5]:
            course_id = rec["course_id"]
            expl = generate_explanation(
                user_id, course_id, user_courses,
                rec, course_concepts, association_rules
            )
            user_explanations.append({
                "course_id": course_id,
                "explanations": expl
            })
        explanations[user_id] = user_explanations

    return recommendations, explanations


## Step 7.7: Evaluation Metrics

In [9]:
def calculate_metrics(recommendations: Dict, explanations: Dict, k: int = 10) -> Dict[str, float]:
    """
    Calculate evaluation metrics

    Args:
        recommendations: Recommendations dict
        explanations: Explanations dict
        k: Top-k parameter

    Returns:
        Dict of metrics
    """
    metrics = {
        "coverage": len(recommendations),
        "avg_recs_per_user": np.mean([len(r) for r in recommendations.values()]),
    }

    # Score distribution
    all_scores = []
    for recs in recommendations.values():
        for rec in recs:
            all_scores.append(rec["final_score"])

    metrics["avg_score"] = np.mean(all_scores)
    metrics["score_std"] = np.std(all_scores)

    # Readiness statistics
    all_readiness = []
    for recs in recommendations.values():
        for rec in recs:
            all_readiness.append(rec["readiness"])

    metrics["avg_readiness"] = np.mean(all_readiness)
    metrics["high_readiness_ratio"] = sum(1 for r in all_readiness if r >= 0.5) / len(all_readiness)

    # Explanation coverage
    total_explanations = 0
    for user_expl in explanations.values():
        for course_expl in user_expl:
            if course_expl["explanations"]:
                total_explanations += 1

    total_recs = sum(len(recs) for recs in recommendations.values())
    metrics["explanation_coverage"] = total_explanations / total_recs if total_recs > 0 else 0

    # Explanation type distribution
    type_counts = defaultdict(int)
    for user_expl in explanations.values():
        for course_expl in user_expl:
            for expl in course_expl["explanations"]:
                type_counts[expl["type"]] += 1

    metrics["path_based_count"] = type_counts.get("path-based", 0)
    metrics["concept_based_count"] = type_counts.get("concept-based", 0)
    metrics["similarity_based_count"] = type_counts.get("similarity-based", 0)

    return metrics

In [10]:

"""Main execution function"""
print("=" * 60)
print("STEP 7: Hybrid Recommendation & Evaluation")
print("=" * 60)

# Load data
user_sequences, course_concepts, course_prereqs, association_rules, \
    user_embeddings, course_embeddings, R_data = load_data()

print("\n" + "-" * 60)

# Generate recommendations
recommendations, explanations = generate_recommendations(
    user_sequences, course_concepts,
    user_embeddings, course_embeddings,
    association_rules, R_data,
    course_prereqs,
    sample_size=1000,
    top_k=10
)

print(f"\nGenerated recommendations for {len(recommendations)} users")

# Calculate metrics
metrics = calculate_metrics(recommendations, explanations)

print("\n" + "-" * 60)
print("Evaluation Metrics:")
print("=" * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# Show example
print("\n" + "-" * 60)
print("Example Recommendations:")
print("=" * 50)
example_user = list(recommendations.keys())[0]
print(f"\nUser: {example_user}")
for i, rec in enumerate(recommendations[example_user][:5], 1):
    print(f"\n  {i}. Course: {rec['course_id']}")
    print(f"     Final Score: {rec['final_score']:.4f}")
    print(f"     Components:")
    print(f"       - ALS: {rec['als_score']:.4f}")
    print(f"       - Rule: {rec['rule_confidence']:.4f}")
    print(f"       - R_score: {rec['r_score']:.4f}")
    print(f"       - Readiness: {rec['readiness']:.4f}")

    # Show explanations
    user_expl = explanations.get(example_user, [])
    if user_expl:
        course_expl = next((e for e in user_expl if e["course_id"] == rec["course_id"]), None)
        if course_expl and course_expl["explanations"]:
            print(f"     Explanations:")
            for expl in course_expl["explanations"]:
                print(f"       [{expl['type']}] {expl['text']}")



STEP 7: Hybrid Recommendation & Evaluation
Loading data...
  - User sequences: 11385
  - Course concepts: 887
  - Course prerequisites: 179
  - Association rules: 327
  - User embeddings: 11385
  - Course embeddings: 744
  - Warning: R_matrix not found, will use default R_score

------------------------------------------------------------
Total courses: 887
Generating recommendations for 1000 users...
  Processing user 20/1000...
  Processing user 40/1000...
  Processing user 60/1000...
  Processing user 80/1000...
  Processing user 100/1000...
  Processing user 120/1000...
  Processing user 140/1000...
  Processing user 160/1000...
  Processing user 180/1000...
  Processing user 200/1000...
  Processing user 220/1000...
  Processing user 240/1000...
  Processing user 260/1000...
  Processing user 280/1000...
  Processing user 300/1000...
  Processing user 320/1000...
  Processing user 340/1000...
  Processing user 360/1000...
  Processing user 380/1000...
  Processing user 400/1000...

## Save Final Results

In [11]:
# Save results
print("\n" + "-" * 60)
print("Saving Results...")
print("=" * 50)

final_output = {
    "recommendations": recommendations,
    "explanations": explanations,
    "metrics": metrics,
    "parameters": {
        "W_ALS": W_ALS,
        "W_RULE": W_RULE,
        "W_RSCORE": W_RSCORE,
        "W_READINESS": W_READINESS,
        "READINESS_THRESHOLD": READINESS_THRESHOLD,
        "READINESS_PENALTY": READINESS_PENALTY
    }
}

# Save as pickle
with open(f"{OUTPUT_PATH}/final_recommendations.pkl", "wb") as f:
    pickle.dump(final_output, f)

# Save as JSON
with open(f"{OUTPUT_PATH}/final_recommendations.json", "w") as f:
    json.dump(final_output, f, indent=2)

print(f"  - Saved: {OUTPUT_PATH}/final_recommendations.pkl")
print(f"  - Saved: {OUTPUT_PATH}/final_recommendations.json")



------------------------------------------------------------
Saving Results...
  - Saved: C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output/final_recommendations.pkl
  - Saved: C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output/final_recommendations.json


## Summary

In [12]:
print("\n" + "=" * 60)
print("STEP 7 COMPLETE - ALL PHASES COMPLETE!")
print("=" * 60)
print(f"\nFinal Results:")
print(f"  - Recommendations: {len(recommendations)} users")
print(f"  - Avg score: {metrics['avg_score']:.4f}")
print(f"  - Avg readiness: {metrics['avg_readiness']:.4f}")
print(f"  - High readiness ratio: {metrics['high_readiness_ratio']:.2%}")
print(f"  - Explanation coverage: {metrics['explanation_coverage']:.2%}")
print(f"\nAll outputs saved to: {OUTPUT_PATH}")


STEP 7 COMPLETE - ALL PHASES COMPLETE!

Final Results:
  - Recommendations: 1000 users
  - Avg score: 0.2802
  - Avg readiness: 0.0809
  - High readiness ratio: 14.49%
  - Explanation coverage: 44.74%

All outputs saved to: C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output
